# Step 3C.1 - Dev Dense Pareto Sweep

This notebook stays on `dev_tune_200` and keeps the **subject gate** fixed.

It adds dense guidance-scale points:

```text
1.25, 1.5, 1.75
```

for:

```text
myopic_score_gated_subject
no_rollout_bridge_gated_subject
mc_bridge_gated_subject
```

It does not touch `analysis_500`, `ablation_500`, `final_test_500`, or `final_test_full`.

In [ ]:
%%bash
set -euo pipefail

pip install -q   "transformers==4.46.3"   "datasets>=4.0.0"   "accelerate>=1.11.0"   "sentencepiece>=0.2.0"   "packaging"   "bitsandbytes>=0.43.0"

In [ ]:
import subprocess
import torch, transformers, datasets, accelerate

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)

assert torch.cuda.is_available(), "This notebook needs a GPU runtime."
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Save Runtime Environment

In [ ]:
import json
from pathlib import Path
import subprocess
import torch, transformers, datasets, accelerate

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert project.exists(), f"Project directory missing after Drive mount: {project}"
try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=project, text=True).strip()
except Exception as exc:
    commit = f"unavailable: {exc!r}"

env = {
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "git_commit": commit,
}
out = project / "runs/counterfact_direction1_v1/runtime_environment_step3c1.json"
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", encoding="utf-8") as f:
    json.dump(env, f, indent=2)
print("Wrote:", out)
print(json.dumps(env, indent=2))

## Preflight

In [ ]:
from pathlib import Path
import json

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert PROJECT_DIR.exists(), f"Missing project dir: {PROJECT_DIR}"

required_files = [
    "llada_runtime_editor_eval.py",
    "llada_experiment_reports.py",
    "runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_gated_v1/report_summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_matched_kl_sweep_subject_v1/report_summary.json",
]
for name in required_files:
    path = PROJECT_DIR / name
    assert path.exists(), f"Missing required file: {path}"

reused_run_dirs = [
    "dev_tune_200_sweep_bridge_controls_subject_gs025",
    "dev_tune_200_sweep_bridge_controls_subject_gs050",
    "dev_tune_200_gated_bridge_controls_subject",
    "dev_tune_200_sweep_bridge_controls_subject_gs200",
    "dev_tune_200_sweep_mc_bridge_subject_gs025",
    "dev_tune_200_sweep_mc_bridge_subject_gs050",
    "dev_tune_200_gated_mc_bridge_subject",
    "dev_tune_200_sweep_mc_bridge_subject_gs200",
    "dev_tune_200_gated_prompt_memory_subject",
]
for run_name in reused_run_dirs:
    run_dir = PROJECT_DIR / "runs/counterfact_direction1_v1" / run_name
    for artifact in ["run_config.json", "summary.json", "per_case_results.jsonl"]:
        path = run_dir / artifact
        assert path.exists(), f"Missing reused Step 3B/3C artifact: {path}"

with (PROJECT_DIR / "runs/counterfact_direction1_v1/dev_tune_200_matched_kl_sweep_subject_v1/report_summary.json").open() as f:
    step3c_report = json.load(f)
assert step3c_report["split_role"] == "dev_tune_200"
assert step3c_report["gate_mode"] == "subject"
assert step3c_report["selfloc_base"] > 0
print("Step 3C prerequisite and reused run artifacts OK.")

## Overwrite Guards

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
run_dirs = [
    project / "runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_bridge_controls_subject_gs125",
    project / "runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_bridge_controls_subject_gs150",
    project / "runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_bridge_controls_subject_gs175",
    project / "runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_mc_bridge_subject_gs125",
    project / "runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_mc_bridge_subject_gs150",
    project / "runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_mc_bridge_subject_gs175",
    project / "runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_subject_v1",
]
for d in run_dirs:
    if d.exists():
        raise RuntimeError(
            f"Run directory already exists: {d}. "
            "Delete it manually or create a new run name. Do not overwrite silently."
        )
print("Overwrite guard passed.")

## Bridge Controls, guidance_scale = 1.25

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_bridge_controls_subject_gs125

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_bridge_controls_subject_gs125 \
  --methods myopic_score_gated_subject,no_rollout_bridge_gated_subject \
  --eval_samples 4 \
  --steps 4 \
  --bridge_topk 4 \
  --mc_rollouts 0 \
  --guidance_scale 1.25 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --gate_mode subject \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_bridge_controls_subject_gs125/stdout.log

## Bridge Controls, guidance_scale = 1.5

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_bridge_controls_subject_gs150

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_bridge_controls_subject_gs150 \
  --methods myopic_score_gated_subject,no_rollout_bridge_gated_subject \
  --eval_samples 4 \
  --steps 4 \
  --bridge_topk 4 \
  --mc_rollouts 0 \
  --guidance_scale 1.5 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --gate_mode subject \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_bridge_controls_subject_gs150/stdout.log

## Bridge Controls, guidance_scale = 1.75

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_bridge_controls_subject_gs175

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_bridge_controls_subject_gs175 \
  --methods myopic_score_gated_subject,no_rollout_bridge_gated_subject \
  --eval_samples 4 \
  --steps 4 \
  --bridge_topk 4 \
  --mc_rollouts 0 \
  --guidance_scale 1.75 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --gate_mode subject \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_bridge_controls_subject_gs175/stdout.log

## MC Bridge, guidance_scale = 1.25

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_mc_bridge_subject_gs125

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_mc_bridge_subject_gs125 \
  --methods mc_bridge_gated_subject \
  --eval_samples 4 \
  --steps 4 \
  --bridge_topk 4 \
  --mc_rollouts 2 \
  --guidance_scale 1.25 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --gate_mode subject \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_mc_bridge_subject_gs125/stdout.log

## MC Bridge, guidance_scale = 1.5

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_mc_bridge_subject_gs150

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_mc_bridge_subject_gs150 \
  --methods mc_bridge_gated_subject \
  --eval_samples 4 \
  --steps 4 \
  --bridge_topk 4 \
  --mc_rollouts 2 \
  --guidance_scale 1.5 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --gate_mode subject \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_mc_bridge_subject_gs150/stdout.log

## MC Bridge, guidance_scale = 1.75

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_mc_bridge_subject_gs175

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_mc_bridge_subject_gs175 \
  --methods mc_bridge_gated_subject \
  --eval_samples 4 \
  --steps 4 \
  --bridge_topk 4 \
  --mc_rollouts 2 \
  --guidance_scale 1.75 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --gate_mode subject \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_mc_bridge_subject_gs175/stdout.log

## Validate Dense Sweep Runs

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
expected_runs = {
    "dev_tune_200_dense_pareto_bridge_controls_subject_gs125": (["myopic_score_gated_subject", "no_rollout_bridge_gated_subject"], 1.25, 0),
    "dev_tune_200_dense_pareto_bridge_controls_subject_gs150": (["myopic_score_gated_subject", "no_rollout_bridge_gated_subject"], 1.5, 0),
    "dev_tune_200_dense_pareto_bridge_controls_subject_gs175": (["myopic_score_gated_subject", "no_rollout_bridge_gated_subject"], 1.75, 0),
    "dev_tune_200_dense_pareto_mc_bridge_subject_gs125": (["mc_bridge_gated_subject"], 1.25, 2),
    "dev_tune_200_dense_pareto_mc_bridge_subject_gs150": (["mc_bridge_gated_subject"], 1.5, 2),
    "dev_tune_200_dense_pareto_mc_bridge_subject_gs175": (["mc_bridge_gated_subject"], 1.75, 2),
}

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

for run_name, (methods, scale, mc_rollouts) in expected_runs.items():
    run_dir = project / "runs/counterfact_direction1_v1" / run_name
    cfg = json.load((run_dir / "run_config.json").open())
    assert cfg["protocol_version"] == "counterfact_direction1_v1"
    assert cfg["split_role"] == "dev_tune_200"
    assert cfg["methods"] == methods
    assert cfg["rollout_config"]["gate_mode"] == "subject"
    assert abs(float(cfg["rollout_config"]["guidance_scale"]) - scale) < 1e-8
    assert int(cfg["rollout_config"]["mc_rollouts"]) == mc_rollouts
    assert "analysis_500" not in str(cfg)
    assert "final_test" not in str(cfg)

    rows = list(read_jsonl(run_dir / "per_case_results.jsonl"))
    by_method_bucket = defaultdict(set)
    for row in rows:
        method = row.get("method_variant") or row.get("method")
        if method in methods and row.get("bucket") in {"rewrite", "declarative_paraphrases", "near_locality", "far_locality"}:
            by_method_bucket[(method, row["bucket"])].add(str(row.get("edit_id") or row.get("case_id")))
    for method in methods:
        for bucket in ["rewrite", "declarative_paraphrases", "near_locality", "far_locality"]:
            n = len(by_method_bucket[(method, bucket)])
            assert n == 200, f"{run_name}/{method}/{bucket}: expected 200 edits, got {n}"
print("Dense sweep configs and completeness checks OK.")

## Build Dense Pareto Report

In [ ]:
import csv
import json
import sys
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
out_dir = project / "runs/counterfact_direction1_v1/dev_tune_200_dense_pareto_subject_v1"
out_dir.mkdir(parents=True, exist_ok=True)
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
from llada_experiment_reports import paired_bootstrap_delta_by_case

thresholds = json.load((project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json").open())
selfloc_base = float(thresholds["selfloc_base"])
near_tfpr_budget = float(thresholds["target_false_positive_rate_budget_by_bucket"]["near_locality"])
far_tfpr_budget = float(thresholds["target_false_positive_rate_budget_by_bucket"]["far_locality"])
malformed_budget = float(thresholds["malformed_span_rate_budget"])
gpu_budget = float(thresholds["gpu_minutes_per_edit_budget"])

run_specs = [
    ("prompt_memory_gated_subject_gs1.0", "prompt_memory", 1.0, "dev_tune_200_gated_prompt_memory_subject", "prompt_memory_gated_subject"),
    ("myopic_score_gated_subject_gs0.25", "myopic_score", 0.25, "dev_tune_200_sweep_bridge_controls_subject_gs025", "myopic_score_gated_subject"),
    ("myopic_score_gated_subject_gs0.5", "myopic_score", 0.5, "dev_tune_200_sweep_bridge_controls_subject_gs050", "myopic_score_gated_subject"),
    ("myopic_score_gated_subject_gs1.0", "myopic_score", 1.0, "dev_tune_200_gated_bridge_controls_subject", "myopic_score_gated_subject"),
    ("myopic_score_gated_subject_gs1.25", "myopic_score", 1.25, "dev_tune_200_dense_pareto_bridge_controls_subject_gs125", "myopic_score_gated_subject"),
    ("myopic_score_gated_subject_gs1.5", "myopic_score", 1.5, "dev_tune_200_dense_pareto_bridge_controls_subject_gs150", "myopic_score_gated_subject"),
    ("myopic_score_gated_subject_gs1.75", "myopic_score", 1.75, "dev_tune_200_dense_pareto_bridge_controls_subject_gs175", "myopic_score_gated_subject"),
    ("myopic_score_gated_subject_gs2.0", "myopic_score", 2.0, "dev_tune_200_sweep_bridge_controls_subject_gs200", "myopic_score_gated_subject"),
    ("no_rollout_bridge_gated_subject_gs0.25", "no_rollout_bridge", 0.25, "dev_tune_200_sweep_bridge_controls_subject_gs025", "no_rollout_bridge_gated_subject"),
    ("no_rollout_bridge_gated_subject_gs0.5", "no_rollout_bridge", 0.5, "dev_tune_200_sweep_bridge_controls_subject_gs050", "no_rollout_bridge_gated_subject"),
    ("no_rollout_bridge_gated_subject_gs1.0", "no_rollout_bridge", 1.0, "dev_tune_200_gated_bridge_controls_subject", "no_rollout_bridge_gated_subject"),
    ("no_rollout_bridge_gated_subject_gs1.25", "no_rollout_bridge", 1.25, "dev_tune_200_dense_pareto_bridge_controls_subject_gs125", "no_rollout_bridge_gated_subject"),
    ("no_rollout_bridge_gated_subject_gs1.5", "no_rollout_bridge", 1.5, "dev_tune_200_dense_pareto_bridge_controls_subject_gs150", "no_rollout_bridge_gated_subject"),
    ("no_rollout_bridge_gated_subject_gs1.75", "no_rollout_bridge", 1.75, "dev_tune_200_dense_pareto_bridge_controls_subject_gs175", "no_rollout_bridge_gated_subject"),
    ("no_rollout_bridge_gated_subject_gs2.0", "no_rollout_bridge", 2.0, "dev_tune_200_sweep_bridge_controls_subject_gs200", "no_rollout_bridge_gated_subject"),
    ("mc_bridge_gated_subject_gs0.25", "mc_bridge", 0.25, "dev_tune_200_sweep_mc_bridge_subject_gs025", "mc_bridge_gated_subject"),
    ("mc_bridge_gated_subject_gs0.5", "mc_bridge", 0.5, "dev_tune_200_sweep_mc_bridge_subject_gs050", "mc_bridge_gated_subject"),
    ("mc_bridge_gated_subject_gs1.0", "mc_bridge", 1.0, "dev_tune_200_gated_mc_bridge_subject", "mc_bridge_gated_subject"),
    ("mc_bridge_gated_subject_gs1.25", "mc_bridge", 1.25, "dev_tune_200_dense_pareto_mc_bridge_subject_gs125", "mc_bridge_gated_subject"),
    ("mc_bridge_gated_subject_gs1.5", "mc_bridge", 1.5, "dev_tune_200_dense_pareto_mc_bridge_subject_gs150", "mc_bridge_gated_subject"),
    ("mc_bridge_gated_subject_gs1.75", "mc_bridge", 1.75, "dev_tune_200_dense_pareto_mc_bridge_subject_gs175", "mc_bridge_gated_subject"),
    ("mc_bridge_gated_subject_gs2.0", "mc_bridge", 2.0, "dev_tune_200_sweep_mc_bridge_subject_gs200", "mc_bridge_gated_subject"),
]

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def mean_or_none(values):
    vals = [float(v) for v in values if v is not None]
    return sum(vals) / len(vals) if vals else None

def harmonic_mean(values, eps=1e-12):
    vals = [max(float(v), eps) for v in values]
    return len(vals) / sum(1.0 / v for v in vals)

def write_csv(path, rows):
    if not rows:
        path.write_text("")
        return
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def aggregate(rows, method):
    filtered = [r for r in rows if (r.get("method_variant") or r.get("method")) == method]
    grouped = defaultdict(list)
    for row in filtered:
        grouped[(row["bucket"], str(row.get("edit_id") or row.get("case_id")))].append(row)
    metric_lists = defaultdict(lambda: defaultdict(list))
    target_len_rows = []
    for (bucket, edit_id), items in grouped.items():
        sample = items[0]
        metrics = {
            "exact_rate": mean_or_none(x.get("exact_rate") for x in items),
            "token_f1": mean_or_none(x.get("token_f1") for x in items),
            "malformed_rate": mean_or_none(x.get("malformed_rate") for x in items),
            "target_false_positive_rate": mean_or_none(x.get("target_false_positive_rate") for x in items),
            "sparse_guidance_kl": mean_or_none(x.get("sparse_guidance_kl") for x in items),
            "gate_activation_rate": mean_or_none(x.get("gate_activation_rate") for x in items),
        }
        for key, value in metrics.items():
            if value is not None:
                metric_lists[bucket][key].append(value)
                target_len_rows.append({"target_length_bin": str(sample.get("target_length_bin")), "bucket": bucket, "metric": key, "value": value, "edit_id": edit_id})
    bucket_summary = {}
    for bucket, values in metric_lists.items():
        bucket_summary[bucket] = {key: mean_or_none(metric_values) for key, metric_values in values.items()}
        bucket_summary[bucket]["num_edits"] = len({str(r.get("edit_id") or r.get("case_id")) for r in filtered if r.get("bucket") == bucket})
    return bucket_summary, target_len_rows

cache = {}
spec_lookup = {label: (label, family, scale, run_name, method) for label, family, scale, run_name, method in run_specs}
summary_rows = []
constraint_rows = []
target_length_raw = []

for label, family, scale, run_name, method in run_specs:
    run_dir = project / "runs/counterfact_direction1_v1" / run_name
    rows = list(read_jsonl(run_dir / "per_case_results.jsonl"))
    cache[run_name] = rows
    run_summary = json.load((run_dir / "summary.json").open())
    bucket_summary, target_len = aggregate(rows, method)
    for bucket in ["rewrite", "declarative_paraphrases", "near_locality", "far_locality"]:
        assert bucket_summary[bucket]["num_edits"] == 200, (label, bucket, bucket_summary[bucket]["num_edits"])

    rewrite = bucket_summary["rewrite"].get("exact_rate")
    paraphrase = bucket_summary["declarative_paraphrases"].get("exact_rate")
    locality = mean_or_none([bucket_summary["near_locality"].get("exact_rate"), bucket_summary["far_locality"].get("exact_rate")])
    clipped = min(locality / selfloc_base, 1.0)
    score = harmonic_mean([rewrite, paraphrase, clipped])
    near_tfpr = bucket_summary["near_locality"].get("target_false_positive_rate")
    far_tfpr = bucket_summary["far_locality"].get("target_false_positive_rate")
    max_malformed = max((bucket_summary[b].get("malformed_rate") or 0.0) for b in bucket_summary)
    primary_kl = mean_or_none([bucket_summary["rewrite"].get("sparse_guidance_kl"), bucket_summary["declarative_paraphrases"].get("sparse_guidance_kl")])
    efficiency = run_summary.get("efficiency", {})
    method_count = len({r.get("method_variant") for r in rows if r.get("method_variant")}) or 1
    gpu_minutes = (float(efficiency.get("wall_time_seconds") or 0.0) / 60.0 / 200.0 / method_count) if efficiency.get("wall_time_seconds") else None
    violations = []
    if near_tfpr is not None and near_tfpr > near_tfpr_budget:
        violations.append("near_tfpr")
    if far_tfpr is not None and far_tfpr > far_tfpr_budget:
        violations.append("far_tfpr")
    if max_malformed > malformed_budget:
        violations.append("malformed")
    if gpu_minutes is None:
        violations.append("gpu_minutes_missing")
    elif gpu_minutes > gpu_budget:
        violations.append("gpu_minutes")
    constraint_pass = not violations
    row = {
        "label": label,
        "family": family,
        "guidance_scale": scale,
        "rewrite_exact": rewrite,
        "declarative_paraphrases_exact": paraphrase,
        "locality_exact": locality,
        "clipped_self_normalized_locality": clipped,
        "selection_score": score,
        "primary_sparse_guidance_kl": primary_kl,
        "near_locality_tfpr": near_tfpr,
        "far_locality_tfpr": far_tfpr,
        "max_malformed_rate": max_malformed,
        "gpu_minutes_per_edit": gpu_minutes,
        "constraint_pass": constraint_pass,
        "constraint_violations": ";".join(violations),
        "feasible_selection_score": score if constraint_pass else None,
    }
    summary_rows.append(row)
    constraint_rows.append({k: row[k] for k in ["label", "family", "guidance_scale", "constraint_pass", "constraint_violations", "near_locality_tfpr", "far_locality_tfpr", "max_malformed_rate", "gpu_minutes_per_edit"]})
    for item in target_len:
        item.update({"label": label, "family": family, "guidance_scale": scale})
        target_length_raw.append(item)

feasible = sorted([r for r in summary_rows if r["constraint_pass"]], key=lambda r: r["feasible_selection_score"], reverse=True)
family_rows = defaultdict(list)
for row in feasible:
    family_rows[row["family"]].append(row)

budgets = [1, 2, 4, 6, 8, 10]
budget_rows = []
guided_budget_rows = []
for budget in budgets:
    eligible = [r for r in feasible if r["primary_sparse_guidance_kl"] is not None and r["primary_sparse_guidance_kl"] <= budget]
    best = max(eligible, key=lambda r: r["feasible_selection_score"], default=None)
    budget_rows.append({"kl_budget": budget, **best} if best else {"kl_budget": budget, "label": None})

    guided_eligible = [r for r in eligible if r["family"] != "prompt_memory"]
    guided_best = max(guided_eligible, key=lambda r: r["feasible_selection_score"], default=None)
    guided_budget_rows.append({"kl_budget": budget, **guided_best} if guided_best else {"kl_budget": budget, "label": None})

def nearest(source, candidates, key):
    valid = [r for r in candidates if r.get(key) is not None]
    if source.get(key) is None or not valid:
        return None
    return min(valid, key=lambda r: abs(float(r[key]) - float(source[key])))

def rows_for_label(label):
    _, _, _, run_name, method = spec_lookup[label]
    return [{**r, "method_variant": label} for r in cache[run_name] if (r.get("method_variant") or r.get("method")) == method]

matched_kl_rows = []
matched_rewrite_rows = []
bootstrap_rows = []
comparison_specs = []
for mc in family_rows.get("mc_bridge", []):
    for family in ["myopic_score", "no_rollout_bridge"]:
        match = nearest(mc, family_rows.get(family, []), "primary_sparse_guidance_kl")
        if match:
            row = {
                "match_type": "matched_kl",
                "anchor_label": mc["label"],
                "compare_label": match["label"],
                "compare_family": family,
                "kl_gap": abs(float(mc["primary_sparse_guidance_kl"]) - float(match["primary_sparse_guidance_kl"])),
                "rewrite_delta_anchor_minus_compare": float(mc["rewrite_exact"]) - float(match["rewrite_exact"]),
                "paraphrase_delta_anchor_minus_compare": float(mc["declarative_paraphrases_exact"]) - float(match["declarative_paraphrases_exact"]),
                "score_delta_anchor_minus_compare": float(mc["feasible_selection_score"]) - float(match["feasible_selection_score"]),
            }
            matched_kl_rows.append(row)
            comparison_specs.append(row)
    for family in ["prompt_memory", "myopic_score", "no_rollout_bridge"]:
        match = nearest(mc, family_rows.get(family, []), "rewrite_exact")
        if match:
            row = {
                "match_type": "matched_rewrite",
                "anchor_label": mc["label"],
                "compare_label": match["label"],
                "compare_family": family,
                "rewrite_gap": abs(float(mc["rewrite_exact"]) - float(match["rewrite_exact"])),
                "paraphrase_delta_anchor_minus_compare": float(mc["declarative_paraphrases_exact"]) - float(match["declarative_paraphrases_exact"]),
                "locality_delta_anchor_minus_compare": float(mc["locality_exact"]) - float(match["locality_exact"]),
                "score_delta_anchor_minus_compare": float(mc["feasible_selection_score"]) - float(match["feasible_selection_score"]),
            }
            matched_rewrite_rows.append(row)
            comparison_specs.append(row)

for comparison in comparison_specs:
    pair_rows = rows_for_label(comparison["anchor_label"]) + rows_for_label(comparison["compare_label"])
    for bucket, metric in [("rewrite", "exact_rate"), ("declarative_paraphrases", "exact_rate")]:
        boot = paired_bootstrap_delta_by_case(pair_rows, candidate_method=comparison["anchor_label"], baseline_method=comparison["compare_label"], bucket=bucket, metric=metric, samples=1000, seed=0)
        if boot:
            bootstrap_rows.append({
                "match_type": comparison["match_type"],
                "anchor_label": comparison["anchor_label"],
                "compare_label": comparison["compare_label"],
                "bucket": bucket,
                "metric": metric,
                "mean_delta_anchor_minus_compare": boot["mean_delta"],
                "ci_lower": boot["ci_low"],
                "ci_upper": boot["ci_high"],
                "num_edits": boot["num_edits"],
            })

target_length_summary_groups = defaultdict(lambda: defaultdict(list))
for row in target_length_raw:
    group_key = (
        row["label"],
        row["family"],
        row["guidance_scale"],
        row["target_length_bin"],
        row["bucket"],
    )
    target_length_summary_groups[group_key][row["metric"]].append(row["value"])

target_length_summary_rows = []
for (label, family, scale, length_bin, bucket), metric_values in sorted(target_length_summary_groups.items()):
    row = {
        "label": label,
        "family": family,
        "guidance_scale": scale,
        "target_length_bin": length_bin,
        "bucket": bucket,
        "num_edits": len({
            item["edit_id"]
            for item in target_length_raw
            if item["label"] == label
            and item["target_length_bin"] == length_bin
            and item["bucket"] == bucket
        }),
    }
    for metric, values in metric_values.items():
        row[f"mean_{metric}"] = mean_or_none(values)
    target_length_summary_rows.append(row)

write_csv(out_dir / "dense_pareto_summary.csv", summary_rows)
write_csv(out_dir / "feasible_ranking.csv", feasible)
write_csv(out_dir / "best_by_kl_budget.csv", budget_rows)
write_csv(out_dir / "best_guided_by_kl_budget.csv", guided_budget_rows)
write_csv(out_dir / "constraint_summary.csv", constraint_rows)
write_csv(out_dir / "matched_kl_comparisons_dense.csv", matched_kl_rows)
write_csv(out_dir / "matched_rewrite_comparisons_dense.csv", matched_rewrite_rows)
write_csv(out_dir / "matched_bootstrap_dense.csv", bootstrap_rows)
write_csv(out_dir / "target_length_breakdown_dense.csv", target_length_raw)
write_csv(out_dir / "target_length_breakdown_dense_summary.csv", target_length_summary_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = {"prompt_memory": "#1f77b4", "myopic_score": "#ff7f0e", "no_rollout_bridge": "#2ca02c", "mc_bridge": "#d62728"}
for family, rows in family_rows.items():
    rows = sorted(rows, key=lambda r: r["primary_sparse_guidance_kl"] or 0.0)
    axes[0].plot([r["primary_sparse_guidance_kl"] for r in rows], [r["rewrite_exact"] for r in rows], marker="o", label=family, color=colors.get(family))
    axes[1].plot([r["primary_sparse_guidance_kl"] for r in rows], [r["feasible_selection_score"] for r in rows], marker="o", label=family, color=colors.get(family))
axes[0].set_xlabel("Primary sparse guidance KL")
axes[0].set_ylabel("Rewrite exact")
axes[0].set_title("Dense Pareto: Rewrite")
axes[0].legend()
axes[1].set_xlabel("Primary sparse guidance KL")
axes[1].set_ylabel("Feasible selection score")
axes[1].set_title("Dense Pareto: Score")
axes[1].legend()
fig.tight_layout()
fig.savefig(out_dir / "dense_pareto_plots.png", dpi=200)
plt.close(fig)

report = {
    "protocol_version": "counterfact_direction1_v1",
    "split_role": "dev_tune_200",
    "gate_mode": "subject",
    "selfloc_base": selfloc_base,
    "kl_budgets": budgets,
    "artifacts": {
        "dense_pareto_summary": str(out_dir / "dense_pareto_summary.csv"),
        "feasible_ranking": str(out_dir / "feasible_ranking.csv"),
        "best_by_kl_budget": str(out_dir / "best_by_kl_budget.csv"),
        "best_guided_by_kl_budget": str(out_dir / "best_guided_by_kl_budget.csv"),
        "constraint_summary": str(out_dir / "constraint_summary.csv"),
        "matched_kl_comparisons_dense": str(out_dir / "matched_kl_comparisons_dense.csv"),
        "matched_rewrite_comparisons_dense": str(out_dir / "matched_rewrite_comparisons_dense.csv"),
        "matched_bootstrap_dense": str(out_dir / "matched_bootstrap_dense.csv"),
        "target_length_breakdown_dense": str(out_dir / "target_length_breakdown_dense.csv"),
        "target_length_breakdown_dense_summary": str(out_dir / "target_length_breakdown_dense_summary.csv"),
        "dense_pareto_plots": str(out_dir / "dense_pareto_plots.png"),
    },
}
with (out_dir / "report_summary.json").open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print("Wrote:", out_dir)
print("Top feasible rows:")
for row in feasible[:10]:
    print(row)
print("Best by KL budget:")
for row in budget_rows:
    print(row)

## Expected Outcome

Inspect first:

```text
feasible_ranking.csv
best_by_kl_budget.csv
best_guided_by_kl_budget.csv
matched_kl_comparisons_dense.csv
matched_bootstrap_dense.csv
dense_pareto_plots.png
target_length_breakdown_dense_summary.csv
```

If MC bridge is competitive at fixed KL budgets after adding the intermediate myopic points, the bridge story stays alive. If myopic remains above MC bridge across the dense frontier, the dev-selected method should probably become myopic scoring unless the stress test changes the safety picture.